# Classification of Titanic passengers

- Let's load the dataset

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

titanic_df = pd.read_csv('titanic.csv', index_col='PassengerId')
titanic_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



## Data preparation

#### Data processing

Let's process the data according to the EDA from the 1st exercise.

In [2]:
from data_processing import process_data

titanic_df = process_data(titanic_df)
titanic_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,2.110213,NoCabin,0,0,1
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,4.280593,C85,1,0,0
3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,2.188856,NoCabin,0,0,1
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,3.990834,C123,0,0,1
5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,2.202765,NoCabin,0,0,1


#### Feature engineering

Let's extract the features(Title, FamilySize, HasCabin, Deck) and remove remaining columns used to extract them

In [3]:
from feature_engineering import engineer_features

titanic_df = engineer_features(titanic_df)
titanic_df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,FamilySize,HasCabin,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_N,Deck_T
PassengerId,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0,3,0,22.0,1,0,2.110213,0,0,1,0,0,1,0,0,2,0,0,0,0,0,0,0,0,1,0
2,1,1,1,38.0,1,0,4.280593,1,0,0,0,0,0,1,0,2,1,0,0,1,0,0,0,0,0,0
3,1,3,1,26.0,0,0,2.188856,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0
4,1,1,1,35.0,1,0,3.990834,0,0,1,0,0,0,1,0,2,1,0,0,1,0,0,0,0,0,0
5,0,3,0,35.0,0,0,2.202765,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0


#### Train/Test Split & Cross-Validation

We split the data into 80% training and 20% test sets, then use 5-fold stratified cross-validation on the training set for model validation. Every model gets the same folds to ensure fairness.

In [4]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    titanic_df.drop(columns="Survived"),
    titanic_df["Survived"],
    test_size=0.2,
    stratify=titanic_df["Survived"],
    random_state=42
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(skf.split(X_train, y_train))

## Model Training & Evaluation
### Baseline
We will take DummyClassifier and test how it manages.

In [5]:
from model_training.common import print_evaluation_results
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
import numpy as np

dummy = DummyClassifier(strategy="most_frequent")

dummy_scores = cross_val_score(
    dummy,
    X_train,
    y_train,
    cv=folds,
    scoring="accuracy"
)

print("Dummy Classifier Evaluation Results:")
print_evaluation_results(dummy_scores)


Dummy Classifier Evaluation Results:
Mean accuracy: 0.6166
Standard deviation: 0.0031


Overall dummy achieves 61% accuracy using the most_frequent strategy, having low spread. <br>
The uniform strategy performed much worse, so I didn't include it here.

### KNN

Now we will take KNN to see how it performs.

In [6]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn_scores = cross_val_score(
    knn,
    X_train,
    y_train,
    cv=folds,
    scoring="accuracy"
)
print("K-Nearest Neighbors Evaluation Results:")
print_evaluation_results(knn_scores)

K-Nearest Neighbors Evaluation Results:
Mean accuracy: 0.7654
Standard deviation: 0.0237


KNN achieves 76% accuracy so outperforming dummy by 15%, with relatively low spread too.

#### Hyperpamaters

Let's find the best-performing hyperparameter combination:

In [7]:
from model_training.common import run_hyperparameter_tuning
from sklearn.neighbors import KNeighborsClassifier

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "cosine", "manhattan"]
}

[best_model, best_params, best_score] = run_hyperparameter_tuning(
    KNeighborsClassifier(),
    param_grid,
    X_train,
    y_train,
    folds
)

print("K-Nearest Neighbors Hyperparameter Tuning Results:")
print(f"Best parameters: {best_params}")
print(f"Best cross-validation score: {best_score:.4f}")

print("Score on test set:", best_model.score(X_test, y_test))

K-Nearest Neighbors Hyperparameter Tuning Results:
Best parameters: {'metric': 'cosine', 'n_neighbors': 11, 'weights': 'distance'}
Best cross-validation score: 0.8118
Score on test set: 0.7877094972067039


So actually, the model improved to 81%, having just a little lower score on a test set. There coud have been model-selection biad or just poor test data, but overall it's quite a good score.

The results themselves are quite interesting.

1) So cosine performed best, suggesting that overall similarity between passenger profiles (linked featues) is more useful than absolute feature differences.
2) 11 neighbors provided the best balance, fewer neighbors lacked information, while more neighbors likely introduced some noise and irrelevant passengers. Distance weighting further reduced the influence of distant neighbors.